In [1]:
# ============================================================
# 09_final_summary.ipynb
# 1. Setup and output inventory
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/Users/hyuna/OpenNeuro/ds007116")

DERIVED_DIR = PROJECT_DIR / "derived"
RESULTS_DIR = PROJECT_DIR / "results" / "summary_tables"
FIGURE_DIR = PROJECT_DIR / "figures"

print("PROJECT_DIR :", PROJECT_DIR)
print("DERIVED_DIR :", DERIVED_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("FIGURE_DIR  :", FIGURE_DIR)

print("\n" + "=" * 70)
print("DERIVED FILES")
print("=" * 70)

for p in sorted(DERIVED_DIR.glob("*")):
    if p.is_file():
        print(p.name)

print("\n" + "=" * 70)
print("SUMMARY TABLES")
print("=" * 70)

for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file():
        print(p.name)

print("\n" + "=" * 70)
print("FIGURES")
print("=" * 70)

for p in sorted(FIGURE_DIR.glob("*")):
    if p.is_file():
        print(p.name)

PROJECT_DIR : /Users/hyuna/OpenNeuro/ds007116
DERIVED_DIR : /Users/hyuna/OpenNeuro/ds007116/derived
RESULTS_DIR : /Users/hyuna/OpenNeuro/ds007116/results/summary_tables
FIGURE_DIR  : /Users/hyuna/OpenNeuro/ds007116/figures

DERIVED FILES
brain_covariates_n77.csv
brain_sample_provenance.csv
clustering_input_z.csv
gmm_cluster_assignments_k5.csv
gmm_posterior_probabilities_k5.csv
network_activation_k5.csv
phenotype_merged_long.csv
wholebrain_analysis_mask_n77.nii.gz
wholebrain_cluster_omnibus_F_FDR05_n77.nii.gz
wholebrain_cluster_omnibus_F_n77.nii.gz
wholebrain_cluster_omnibus_p_FDR_n77.nii.gz
wholebrain_cluster_omnibus_p_uncorrected_n77.nii.gz

SUMMARY TABLES
Cont_pairwise_cluster_contrasts_k5.csv
DorsAttn_pairwise_cluster_contrasts_k5.csv
clinical_characterization_k5.csv
clinical_descriptives_k5.csv
diagnostic_counts_k5.csv
diagnostic_mapping_statistics_k5.csv
diagnostic_proportions_k5.csv
gmm_k5_bootstrap_cluster_recovery.csv
gmm_k5_bootstrap_results.csv
gmm_k5_cluster_recovery_summary

In [2]:
# ============================================================
# 2. Load final analysis outputs
# ============================================================

# ---------- GMM ----------
model_selection = pd.read_csv(
    RESULTS_DIR / "gmm_model_selection.csv"
)

phenotype_profiles = pd.read_csv(
    RESULTS_DIR / "gmm_k5_standardized_profiles.csv"
)

posterior_summary = pd.read_csv(
    RESULTS_DIR / "posterior_probability_by_cluster_k5.csv"
)

cluster_assignments = pd.read_csv(
    DERIVED_DIR / "gmm_cluster_assignments_k5.csv"
)

# ---------- Phenotype / clinical ----------
phenotype_characterization = pd.read_csv(
    RESULTS_DIR / "phenotype_characterization_k5.csv"
)

diagnostic_stats = pd.read_csv(
    RESULTS_DIR / "diagnostic_mapping_statistics_k5.csv"
)

clinical_characterization = pd.read_csv(
    RESULTS_DIR / "clinical_characterization_k5.csv"
)

# ---------- Network neuroimaging ----------
network_primary = pd.read_csv(
    RESULTS_DIR / "network_ancova_primary_n77.csv"
)

network_sensitivity = pd.read_csv(
    RESULTS_DIR / "network_ancova_sensitivity_ses1_n71.csv"
)

network_adjusted = pd.read_csv(
    RESULTS_DIR / "network_adjusted_means_primary_n77.csv"
)

network_posthoc = pd.read_csv(
    RESULTS_DIR / "network_posthoc_holm_primary_n77.csv"
)

# ---------- Whole brain ----------
wholebrain_components = pd.read_csv(
    RESULTS_DIR / "wholebrain_top10_components_n77.csv"
)

wholebrain_network_overlap = pd.read_csv(
    RESULTS_DIR / "wholebrain_FDR_voxels_by_Yeo7_network_n77.csv"
)

# ---------- Stability ----------
bootstrap = pd.read_csv(
    RESULTS_DIR / "gmm_k5_bootstrap_results.csv"
)

cluster_recovery = pd.read_csv(
    RESULTS_DIR / "gmm_k5_cluster_recovery_summary.csv"
)

print("All final result tables loaded.")

All final result tables loaded.


In [3]:
# ============================================================
# 3. Sample-size QC
# ============================================================

print("=" * 60)
print("SAMPLE FLOW")
print("=" * 60)

print(
    "Phenotype clustering N:",
    cluster_assignments["participant_id"].nunique()
)

brain_covariates = pd.read_csv(
    DERIVED_DIR / "brain_covariates_n77.csv"
)

print(
    "Primary brain N:",
    brain_covariates["participant_id"].nunique()
)

print(
    "Primary cluster sizes:"
)

print(
    cluster_assignments["cluster"]
    .value_counts()
    .sort_index()
)

SAMPLE FLOW
Phenotype clustering N: 92
Primary brain N: 77
Primary cluster sizes:
cluster
0     6
1    41
2     9
3     7
4    29
Name: count, dtype: int64


In [4]:
# ============================================================
# 4. GMM QC
# ============================================================

print("=" * 60)
print("GMM MODEL SELECTION")
print("=" * 60)

print(model_selection.to_string(index=False))

print("\n" + "=" * 60)
print("POSTERIOR CLASSIFICATION")
print("=" * 60)

print(
    posterior_summary.to_string(index=False)
)

print("\n" + "=" * 60)
print("BOOTSTRAP STABILITY")
print("=" * 60)

print(
    bootstrap["ARI"].describe(
        percentiles=[.05, .50, .95]
    )
)

print("\nCluster recovery:")
print(
    cluster_recovery.to_string(index=False)
)

GMM MODEL SELECTION
 k         BIC         AIC  silhouette  cluster_sizes  converged  n_iter
 2 2772.253727 2381.376497    0.205466           8/84       True       2
 3 2721.062843 2133.486105    0.244308         81/8/3       True       2
 4 2753.401695 1969.125448    0.100613      7/27/53/5       True      19
 5 2707.918528 1726.942771    0.111988    6/41/9/7/29       True       5
 6 2659.775144 1482.099879    0.104152 5/40/5/10/6/26       True      16

POSTERIOR CLASSIFICATION
 cluster  count     mean          std      min   median
       0      6 1.000000 0.000000e+00 1.000000 1.000000
       1     41 0.988915 3.897758e-02 0.754670 0.999980
       2      9 1.000000 1.331035e-09 1.000000 1.000000
       3      7 1.000000 0.000000e+00 1.000000 1.000000
       4     29 0.986386 3.098477e-02 0.876313 0.999658

BOOTSTRAP STABILITY
count    500.000000
mean       0.166308
std        0.085646
min       -0.024262
5%         0.039789
50%        0.158235
95%        0.310424
max        0.459772

In [5]:
print("Clinical columns:")
print(clinical_characterization.columns.tolist())

print("\nDiagnostic columns:")
print(diagnostic_stats.columns.tolist())

Clinical columns:
['outcome', 'N', 'F', 'p_anova', 'eta_squared', 'H', 'p_kw', 'epsilon_squared', 'p_anova_fdr', 'p_kw_fdr']

Diagnostic columns:
['chi_square', 'df', 'p_asymptotic', 'p_permutation', 'cramers_v', 'expected_cells_below_5', 'expected_cells_total', 'percent_expected_below_5']


In [6]:
print("Network columns:")
print(network_primary.columns.tolist())

Network columns:
['network', 'F', 'df_cluster', 'df_residual', 'p', 'partial_eta2', 'p_fdr', 'fdr_significant']


In [7]:
import nibabel as nib

analysis_mask_img = nib.load(
    DERIVED_DIR /
    "wholebrain_analysis_mask_n77.nii.gz"
)

fdr_img = nib.load(
    DERIVED_DIR /
    "wholebrain_cluster_omnibus_F_FDR05_n77.nii.gz"
)

analysis_mask_data = (
    analysis_mask_img.get_fdata() > 0
)

fdr_data = fdr_img.get_fdata()

n_tested = int(
    analysis_mask_data.sum()
)

n_significant = int(
    np.sum(fdr_data > 0)
)

percent_significant = (
    100 * n_significant / n_tested
)

print("=" * 60)
print("WHOLE-BRAIN QC")
print("=" * 60)

print(f"Tested voxels:       {n_tested:,}")
print(f"FDR-significant:     {n_significant:,}")
print(f"Percent significant: {percent_significant:.2f}%")

print("\nTop components:")
print(
    wholebrain_components[
        [
            "component",
            "n_voxels",
            "anatomical_description"
        ]
    ].to_string(index=False)
)

print("\nYeo-7 spatial overlap:")
print(
    wholebrain_network_overlap
    .sort_values(
        "percent_significant",
        ascending=False
    )
    .to_string(index=False)
)

WHOLE-BRAIN QC
Tested voxels:       174,794
FDR-significant:     6,658
Percent significant: 3.81%

Top components:
 component  n_voxels                            anatomical_description
         3      1520                               Left frontal cortex
         9       795 Left lateral occipital / superior parietal cortex
       124       559    Right superior parietal / supramarginal cortex
         1       342                     Left temporo-occipital cortex
       109       291            Right superior / middle frontal cortex
        61       278             Medial frontal / paracingulate cortex
        20       229           Left insular / frontal opercular cortex
       107       223               Right cerebellar Crus I / lobule VI
       119       165          Right insular / frontal opercular cortex
        82       158            Right cerebellar lobule VI / Crus I-II

Yeo-7 spatial overlap:
    network  analysis_voxels  significant_voxels  percent_significant
   DorsAtt

In [8]:
# ============================================================
# 7. Final clinical and diagnostic summary
# ============================================================

clinical_final = clinical_characterization.copy()

clinical_final["anova_fdr_sig"] = (
    clinical_final["p_anova_fdr"] < .05
)

clinical_final["kw_fdr_sig"] = (
    clinical_final["p_kw_fdr"] < .05
)

clinical_final["robust_both"] = (
    clinical_final["anova_fdr_sig"] &
    clinical_final["kw_fdr_sig"]
)

clinical_significant = clinical_final[
    clinical_final["anova_fdr_sig"] |
    clinical_final["kw_fdr_sig"]
].copy()

clinical_significant = clinical_significant[
    [
        "outcome",
        "N",
        "F",
        "p_anova",
        "eta_squared",
        "p_anova_fdr",
        "H",
        "p_kw",
        "epsilon_squared",
        "p_kw_fdr",
        "robust_both"
    ]
]

print("=" * 70)
print("CLINICAL OUTCOMES SURVIVING FDR IN >= 1 TEST")
print("=" * 70)

print(
    clinical_significant
    .sort_values("p_anova_fdr")
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("DIAGNOSTIC MAPPING")
print("=" * 70)

print(
    diagnostic_stats.to_string(index=False)
)

CLINICAL OUTCOMES SURVIVING FDR IN >= 1 TEST
            outcome  N         F      p_anova  eta_squared  p_anova_fdr         H     p_kw  epsilon_squared  p_kw_fdr  robust_both
               MAPS 83 13.910485 1.307376e-08     0.416351 1.699589e-07 22.815164 0.000138         0.241220  0.001792         True
Pre-scan STAI Trait 80  6.197821 2.303261e-04     0.248431 1.497120e-03 21.244447 0.000283         0.229926  0.001841         True
     PANAS Positive 55  4.864382 2.160436e-03     0.280136 9.361888e-03 14.871990 0.004974         0.217440  0.021555         True
Pre-scan STAI State 84  4.124778 4.362566e-03     0.172767 1.417834e-02 11.477372 0.021692         0.094650  0.056399        False
               RPAS 90  3.692751 8.043991e-03     0.148049 2.091438e-02 12.322249 0.015109         0.097909  0.049105         True

DIAGNOSTIC MAPPING
 chi_square  df  p_asymptotic  p_permutation  cramers_v  expected_cells_below_5  expected_cells_total  percent_expected_below_5
  10.018567   8      

In [9]:
# ============================================================
# 8. Final network-level results
# ============================================================

network_final = (
    network_primary
    .sort_values("p_fdr")
    .reset_index(drop=True)
)

print("=" * 70)
print("PRIMARY NETWORK ANALYSIS — N=77")
print("=" * 70)

print(
    network_final.to_string(index=False)
)

print("\nFDR-significant networks:")

print(
    network_final.loc[
        network_final["fdr_significant"],
        [
            "network",
            "F",
            "df_cluster",
            "df_residual",
            "p",
            "partial_eta2",
            "p_fdr"
        ]
    ].to_string(index=False)
)

PRIMARY NETWORK ANALYSIS — N=77
    network        F  df_cluster  df_residual        p  partial_eta2    p_fdr  fdr_significant
   DorsAttn 6.625458         4.0         70.0 0.000140      0.274625 0.000979             True
       Cont 4.985770         4.0         70.0 0.001355      0.221730 0.004744             True
SalVentAttn 2.930610         4.0         70.0 0.026676      0.143442 0.062243            False
        Vis 2.518224         4.0         70.0 0.048872      0.125797 0.082969            False
     Limbic 2.386533         4.0         70.0 0.059264      0.120007 0.082969            False
    Default 2.074362         4.0         70.0 0.093368      0.105973 0.108930            False
     SomMot 1.415791         4.0         70.0 0.237761      0.074847 0.237761            False

FDR-significant networks:
 network        F  df_cluster  df_residual        p  partial_eta2    p_fdr
DorsAttn 6.625458         4.0         70.0 0.000140      0.274625 0.000979
    Cont 4.985770         4.0  

In [10]:
# ============================================================
# 9. Master results summary
# ============================================================

ari_median = bootstrap["ARI"].median()

dors = network_primary.loc[
    network_primary["network"] == "DorsAttn"
].iloc[0]

cont = network_primary.loc[
    network_primary["network"] == "Cont"
].iloc[0]

diag = diagnostic_stats.iloc[0]

master_results = pd.DataFrame([
    {
        "domain": "Phenotype construction",
        "analysis": "GMM",
        "sample": "N=92",
        "result": (
            "Five-profile exploratory solution; "
            "cluster sizes = 6/41/9/7/29"
        )
    },
    {
        "domain": "Diagnostic mapping",
        "analysis": "Chi-square + permutation",
        "sample": "N=92",
        "result": (
            f"Chi-square={diag['chi_square']:.3f}, "
            f"df={int(diag['df'])}, "
            f"permutation p={diag['p_permutation']:.3f}, "
            f"Cramer's V={diag['cramers_v']:.3f}"
        )
    },
    {
        "domain": "Network neuroimaging",
        "analysis": "Dorsal Attention ANCOVA",
        "sample": "N=77",
        "result": (
            f"F({int(dors['df_cluster'])},"
            f"{int(dors['df_residual'])})="
            f"{dors['F']:.3f}, "
            f"FDR p={dors['p_fdr']:.4f}, "
            f"partial eta2={dors['partial_eta2']:.3f}"
        )
    },
    {
        "domain": "Network neuroimaging",
        "analysis": "Control ANCOVA",
        "sample": "N=77",
        "result": (
            f"F({int(cont['df_cluster'])},"
            f"{int(cont['df_residual'])})="
            f"{cont['F']:.3f}, "
            f"FDR p={cont['p_fdr']:.4f}, "
            f"partial eta2={cont['partial_eta2']:.3f}"
        )
    },
    {
        "domain": "Whole-brain neuroimaging",
        "analysis": "Voxelwise omnibus GLM",
        "sample": "N=77",
        "result": (
            f"6,658 / 174,794 voxels "
            f"({100 * 6658 / 174794:.2f}%) "
            f"survived BH-FDR q<.05"
        )
    },
    {
        "domain": "Stability",
        "analysis": "500 bootstrap resamples",
        "sample": "N=92",
        "result": (
            f"Median ARI={ari_median:.3f}; "
            "low resampling stability"
        )
    }
])

pd.set_option(
    "display.max_colwidth",
    None
)

display(master_results)

,domain,analysis,sample,result
0,Phenotype construction,GMM,N=92,Five-profile exploratory solution; cluster sizes = 6/41/9/7/29
1,Diagnostic mapping,Chi-square + permutation,N=92,"Chi-square=10.019, df=8, permutation p=0.270, Cramer's V=0.233"
2,Network neuroimaging,Dorsal Attention ANCOVA,N=77,"F(4,70)=6.625, FDR p=0.0010, partial eta2=0.275"
3,Network neuroimaging,Control ANCOVA,N=77,"F(4,70)=4.986, FDR p=0.0047, partial eta2=0.222"
4,Whole-brain neuroimaging,Voxelwise omnibus GLM,N=77,"6,658 / 174,794 voxels (3.81%) survived BH-FDR q<.05"
5,Stability,500 bootstrap resamples,N=92,Median ARI=0.158; low resampling stability


In [11]:
master_results.to_csv(
    RESULTS_DIR /
    "final_master_results_summary.csv",
    index=False
)

clinical_significant.to_csv(
    RESULTS_DIR /
    "final_clinical_FDR_results.csv",
    index=False
)

network_final.to_csv(
    RESULTS_DIR /
    "final_network_results.csv",
    index=False
)

print("Final summary tables saved.")

Final summary tables saved.
